# Comment Toxicity Detection — Training Notebook

This notebook walks through the full pipeline for training a **BiLSTM deep-learning model** to detect toxic comments across six categories: `toxic`, `severe_toxic`, `obscene`, `threat`, `insult`, and `identity_hate`.

**Steps covered:**
1. Clone the project repository
2. Install required dependencies
3. Train the model (with GPU acceleration)
4. Package model artifacts for download
5. Deploy the Streamlit app via ngrok (optional)

> **Tip:** This notebook is designed to run on **Google Colab** with a T4 GPU runtime for optimal training speed.

## Step 1 — Clone the Repository

Clone the project from GitHub and navigate into the project directory. The repository includes the dataset, source code, and configuration files needed for training.

In [6]:
# Cell 1: Clone the repository
!git clone https://github.com/10Unknownboy/Comment-Toxicity-Detection.git
%cd Comment-Toxicity-Detection

Cloning into 'Comment-Toxicity-Detection'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 36 (delta 2), reused 9 (delta 2), pack-reused 22 (from 1)
Receiving objects: 100% (36/36), 50.00 MiB | 20.33 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/Comment-Toxicity-Detection/Comment-Toxicity-Detection


## Step 2 — Install Dependencies

Install the required Python packages. Most of these come pre-installed on Google Colab, but running this cell ensures all versions are compatible.

In [7]:
# Cell 2: Install dependencies
!pip install torch pandas numpy scikit-learn matplotlib tqdm

## Step 3 — Train the Model

Train the BiLSTM model on the Jigsaw Toxic Comment dataset. This cell:

- Loads and preprocesses the training data
- Builds a vocabulary from the corpus
- Trains the BiLSTM model for up to 20 epochs with early stopping
- Evaluates the best model on the held-out test set
- Tunes per-label decision thresholds to maximise F1 scores
- Saves all model artifacts to the `models/` directory

> **Note:** Using `--sample-size 0` trains on the full dataset (~560K comments). For a faster trial, use `--sample-size 160000`.

In [8]:
# Cell 3: Train the model (uses GPU automatically if available)
!python -m src.train --epochs 20 --batch-size 256 --sample-size 0


  Device : cuda
  GPU    : Tesla T4

[data] Loading training data from /content/Comment-Toxicity-Detection/Comment-Toxicity-Detection/data/train.csv …
[data] Loaded 159,571 rows.
[data] Cleaning texts …
[data] Splits → train: 127,656  val: 15,957  test: 15,958
[data] Building vocabulary …
[data] Vocabulary size: 50,000 tokens (+ PAD, UNK).
[data] Vocabulary saved to /content/Comment-Toxicity-Detection/Comment-Toxicity-Detection/models/vocab.json
[data] Encoding sequences …

[model] Total parameters     : 7,076,550
[model] Trainable parameters : 7,076,550

[train] Calculating class weights from training data…
[train] Positive weights per label:
  toxic                : 9.43
  severe_toxic         : 99.20
  obscene              : 17.96
  threat               : 314.98
  insult               : 19.38
  identity_hate        : 113.90

  Starting training …

Epoch 1/20 [train]: 100% 499/499 [00:48<00:00, 10.33it/s, loss=0.7273]
Epoch 1/20 [val]: 100% 63/63 [00:02<00:00, 25.74it/s]

  Epoch 1/

## Step 4 — Package Model Artifacts

Zip all trained model files into a single archive for easy download. After running this cell, use the Colab file browser (left sidebar) to download `models.zip`.

In [9]:
# Cell 4: Zip model artifacts for easy download
!zip -r models.zip models/

  adding: models/ (stored 0%)
  adding: models/roc_curves.png (deflated 12%)
  adding: models/evaluation_results.json (deflated 64%)
  adding: models/training_history.png (deflated 10%)
  adding: models/toxicity_model.pth (deflated 7%)
  adding: models/confusion_matrices.png (deflated 22%)
  adding: models/training_history.json (deflated 51%)
  adding: models/thresholds.json (deflated 38%)
  adding: models/vocab.json (deflated 61%)


## Step 5 — Deploy with ngrok (Optional)

Launch the Streamlit web application directly from Colab using ngrok to create a public tunnel. This lets you preview the app without a local setup.

> **Important:** Replace the ngrok auth token below with your own token from [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken) if needed.

In [10]:
!pip install streamlit pyngrok --quiet

from pyngrok import ngrok
import threading
import os

# Paste your ngrok auth token here
ngrok.set_auth_token("2tqBfHv7S8tZis37fWjCpjqxk7D_2Xf4LsASPoUweghG6Dptd")

def run():
    os.system("streamlit run app.py --server.port 8501")

threading.Thread(target=run).start()

public_url = ngrok.connect(8501)

print("Your Streamlit App is Live:")
print(public_url)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 122.7 MB/s eta 0:00:00
Your Streamlit App is Live:
NgrokTunnel: "https://de7a-34-125-170-84.ngrok-free.app" -> "http://localhost:8501"
